In [1]:
! pip install -U spacy pdfminer.six python-docx

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached numpy-2.3.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (35 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached click-8.2.1-py3-none-any.whl.metadata (2.5 kB)
  Using cached rich-14.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached wrapt-1.17.2-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached MarkupSafe-3.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
! pip install PyPDF2

In [7]:
! pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 4.7 MB/s eta 0:00:0000:0100:01m


In [2]:
! python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [60]:
import spacy
import re
import fitz

In [61]:
def read_pdf_pymupdf(file_path):
    text = ""
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

In [62]:
text = read_pdf_pymupdf('data/WordPress-Dev.pdf')

In [63]:
text

' \nAadim Prajapati \nI’m a third-year Computer Engineering student with a passion for computer \ntechnologies. I love learning about new stuff and am interested in \ndiscovering new places, things and people. \nBhaktapur, Nepal \n9861886233 \naadimpraz@gmail.com \nLinkedin: \nhttps://www.linkedin.com/in/aadi\nm-prajapati-2a1974230/  \nGit: https://github.com/PR7175Z  \nINVOLVEMENTS \nWordPress Workshop for Beginners— Instructor \n●\u200b\nPresented through a collaborative effort between Collabyte and \nKhwopa Engineering College \nEXPERIENCE \nFnClick, Libali, Bhaktapur— Wordpress Developer \nJune 2020- PRESENT \nProjects \nValley Essence \nValley Essence is a blogging platform developed using PHP, HTML, CSS, and \nJavaScript, designed to facilitate discussions and sharing of content about \nvarious aspects of the Kathmandu Valley. The platform supports a \nhierarchical user role system comprising Guest User, User, Author, and \nAdmin roles, with Guest User holding the most limited pe

In [64]:
nlp = spacy.load('en_core_web_sm')

In [84]:
def get_entities(text):
    doc = nlp(text)
    entities = {
        "PERSON": [],
        "ORG": [],
        "GPE": [],
        "DATE": [],
        "EMAIL": [],
        "PHONE": [],
        "SKILL": [],
        "EXPERIENCE" : [],
        "EDUCATION" : [],
    }
    for ent in doc.ents:
        if ent.label_ not in entities:
            entities[ent.label_] = []
        entities[ent.label_].append(ent.text)
    return entities

In [85]:
def get_email(text):
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    return re.findall(email_pattern, text)

In [86]:
def get_phone(text):
    phone_pattern = r'\+?\d[\d -]{8,}\d'
    return re.findall(phone_pattern, text)

In [87]:
def get_skills(text, skills):
    text = text.lower()
    skills = [skill for skill in skills if skill.lower() in text]
    return list(set(skills))

In [88]:
def get_experience(text):
    text = text.lower()
    experience_pattern = r"""
        experience\s*          
        (.*?)                            
        (?=(?:education|projects|skills|$)) 
    """

    matches = re.findall(experience_pattern, text, re.DOTALL | re.VERBOSE)

    experiences = []
    for match in matches:
        entries = re.findall(
            r"""(?P<company>.+?)\s*[—-]\s*(?P<title>.+?)\s*\n
                (?P<duration>(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s*\d{4}\s*[-–]\s*(?:present|\w+\s*\d{4}))""",
            match,
            re.IGNORECASE | re.VERBOSE
        )
        for entry in entries:
            experiences.append({
                "company": entry[0].strip().title(),
                "title": entry[1].strip().title(),
                "duration": entry[2].strip()
            })

    return experiences


In [89]:
print(get_experience(text))

[{'company': 'Fnclick, Libali, Bhaktapur', 'title': 'Wordpress Developer', 'duration': 'june 2020- present'}]


In [90]:
def get_education(text):
    text = text.lower()
    education_pattern = r"""
        education\s*          
        (.*?)                            
        (?=(?:projects|skills|$)) 
    """

    matches = re.findall(education_pattern, text, re.DOTALL | re.VERBOSE)

    return matches

In [91]:
print(get_education(text))

['bagiswori secondary school, bhaktapur— see, +2 \nkhwopa engineering college (in progress) \n●\u200b purbanchal university, bhaktapur \n●\u200b expected graduation: 2026 \n']


In [92]:
skills = ['Python', 'Java', 'C++', 'JavaScript', 'SQL', 'Machine Learning', 'Data Analysis', 'Web Development', 'Cloud Computing', 'DevOps', 'ReactJS', 'NodeJS', 'Django', 'Flask', 'TensorFlow', 'PyTorch', 'Keras', 'NumPy', 'Pandas', 'Scikit-learn', 'Natural Language Processing', 'Computer Vision', 'HTML', 'CSS', 'Bootstrap', 'Git', 'Docker', 'Kubernetes', 'AWS', 'Azure', 'Google Cloud Platform', 'FastAPI', 'Figma', 'Shopify', 'WordPress']

In [93]:
def get_ner(text, skills):
    entities = get_entities(text)
    email = get_email(text)
    phone = get_phone(text)
    skills = get_skills(text, skills)
    experience = get_experience(text)
    education = get_education(text)

    ner_data = {
        "PERSON": entities.get("PERSON", []),
        "ORG": entities.get("ORG", []),
        "GPE": entities.get("GPE", []),
        "DATE": entities.get("DATE", []),
        "EMAIL": email,
        "PHONE": phone,
        "SKILL": skills,
        "EXPERIENCE": experience,
        "EDUCATION" : education,
        "LANGUAGE": entities.get("LANGUAGE", []),
        "CERTIFICATION": entities.get("CERTIFICATION", []),
        "PROJECT": entities.get("PROJECT", []),
    }

    return ner_data

In [94]:
ner = get_ner(text, skills)

In [95]:
ner['PERSON']

['FnClick', 'Admin', 'Admin', 'Javascript', 'Tools']

In [96]:
ner['ORG']

['Computer Engineering',
 'WordPress',
 'Beginners',
 'Collabyte',
 'Khwopa Engineering College',
 'PHP',
 'HTML',
 'CSS',
 'JavaScript',
 'Guest User',
 'Guest User',
 'LaPasa',
 'WordPress',
 'WooCommerce',
 'LaPasa',
 'Bagiswori Secondary School',
 'Khwopa Engineering College',
 'Purbanchal University',
 'PHP',
 'HTML',
 'CSS',
 'WordPress',
 'Figma']

In [97]:
ner['GPE']

['Prajapati',
 'Bhaktapur',
 'Nepal',
 'Libali',
 'Bhaktapur',
 'Bhaktapur',
 'Bhaktapur',
 'Shopify']

In [98]:
ner['DATE']

['third-year', '9861886233', 'June 2020-', '2026']

In [99]:
ner['EMAIL']

['aadimpraz@gmail.com']

In [100]:
ner['PHONE']

['9861886233']

In [101]:
ner['SKILL']

['CSS',
 'Figma',
 'JavaScript',
 'Java',
 'ReactJS',
 'Shopify',
 'Web Development',
 'HTML',
 'Git',
 'WordPress',
 'NodeJS']

In [102]:
ner['EXPERIENCE']

[{'company': 'Fnclick, Libali, Bhaktapur',
  'title': 'Wordpress Developer',
  'duration': 'june 2020- present'}]

In [103]:
ner['EDUCATION']

['bagiswori secondary school, bhaktapur— see, +2 \nkhwopa engineering college (in progress) \n●\u200b purbanchal university, bhaktapur \n●\u200b expected graduation: 2026 \n']